# 02 — Defunciones Generales INEC: limpieza y extracción

**Fuente:** Registros de Defunciones Generales — INEC  
**Archivos:** CSV por año 2019–2024 en `raw/INEC/`  
**Objetivo:** Unir los 6 archivos, filtrar muertes por diabetes (CIE-10 E10–E14), y agregar por provincia y año para el Cruce 1 (mortalidad vs. insulina SERCOP).

In [1]:
import pandas as pd
import unicodedata
from pathlib import Path

pd.set_option('display.float_format', '{:,.2f}'.format)

RAW_DIR = Path('../raw/INEC')
PROCESSED_DIR = Path('../processed')
PROCESSED_DIR.mkdir(exist_ok=True)

ARCHIVOS = {
    2019: 'BDD_EDG_2019.csv',
    2020: 'EDG_2020_CSV_v1.csv',
    2021: 'EDG_2021_CSV.csv',
    2022: 'EDG_2022_CSV_versión_final.csv',
    2023: 'edg_2023_csv.csv',
    2024: 'EDG_2024_CSV.csv',
}

PROVINCIAS = {
    1: 'AZUAY', 2: 'BOLIVAR', 3: 'CAÑAR', 4: 'CARCHI', 5: 'CHIMBORAZO',
    6: 'COTOPAXI', 7: 'EL ORO', 8: 'ESMERALDAS', 9: 'GUAYAS', 10: 'IMBABURA',
    11: 'LOJA', 12: 'LOS RIOS', 13: 'MANABI', 14: 'MORONA SANTIAGO', 15: 'NAPO',
    16: 'PASTAZA', 17: 'PICHINCHA', 18: 'TUNGURAHUA', 19: 'ZAMORA CHINCHIPE',
    20: 'GALAPAGOS', 21: 'SUCUMBIOS', 22: 'ORELLANA',
    23: 'SANTO DOMINGO DE LOS TSACHILAS', 24: 'SANTA ELENA'
}

def quitar_tildes(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

# Mapa inverso: nombre normalizado (sin tildes, uppercase) → código numérico
NOMBRE_A_COD = {
    quitar_tildes(nombre).upper(): cod
    for cod, nombre in PROVINCIAS.items()
}

def normalizar_prov(serie):
    """Convierte prov_res a código numérico sin importar si viene como int o nombre."""
    num = pd.to_numeric(serie, errors='coerce')
    # Para los que no son numéricos, mapear por nombre
    es_texto = num.isna() & serie.notna()
    nombres_norm = serie[es_texto].astype(str).apply(
        lambda x: quitar_tildes(x.strip()).upper()
    ).map(NOMBRE_A_COD)
    num[es_texto] = nombres_norm
    return num

## 1. Carga y unión de los 6 archivos

In [2]:
dfs = []
for year, fname in ARCHIVOS.items():
    df = pd.read_csv(
        RAW_DIR / fname,
        sep=';',
        encoding='latin-1',
        low_memory=False
    )
    # Normalizar nombre de primera columna (viene corrupto por BOM + encoding)
    df.columns = ['numeracion'] + list(df.columns[1:])
    # prov_res: en 2019 es código numérico; en 2020-2024 es nombre de provincia
    df['prov_res'] = normalizar_prov(df['prov_res'])
    df['area_res'] = pd.to_numeric(df['area_res'], errors='coerce')
    df['anio'] = year
    dfs.append(df)
    print(f'{year}: {len(df):>6} defunciones | cols: {df.shape[1]}')

df_all = pd.concat(dfs, ignore_index=True)
print(f'\nTotal unificado: {len(df_all):,} defunciones')

2019:  75355 defunciones | cols: 46


2020: 117030 defunciones | cols: 45


2021: 107648 defunciones | cols: 46


2022:  91954 defunciones | cols: 46


2023:  89877 defunciones | cols: 46


2024:  91803 defunciones | cols: 46

Total unificado: 573,667 defunciones


## 2. Inspección inicial

In [3]:
# Verificar campos clave
print('causa (muestra):', df_all['causa'].dropna().head(8).tolist())
print('area_res valores:', df_all['area_res'].value_counts().to_dict())
print('prov_res nulos:', df_all['prov_res'].isna().sum())

causa (muestra): ['V09', 'B20', 'X59', 'I61', 'I51', 'J18', 'B24', 'B24']
area_res valores: {1.0: 57864, 2.0: 17491}
prov_res nulos: 108371


In [4]:
# Total defunciones por año
df_all.groupby('anio').size().rename('total_defunciones')

anio
2019     75355
2020    117030
2021    107648
2022     91954
2023     89877
2024     91803
Name: total_defunciones, dtype: int64

## 3. Filtro CIE-10 E10–E14 (diabetes mellitus)

| Código | Descripción |
|--------|-------------|
| E10 | Diabetes mellitus insulinodependiente |
| E11 | Diabetes mellitus no insulinodependiente (tipo 2) |
| E12 | Diabetes mellitus asociada con desnutrición |
| E13 | Otras formas especificadas de diabetes mellitus |
| E14 | Diabetes mellitus no especificada |

In [5]:
mask_diabetes = df_all['causa'].astype(str).str.match(r'^E1[0-4]', na=False)
df_db = df_all[mask_diabetes].copy()

print(f'Defunciones por diabetes (E10-E14): {len(df_db):,}')
print(f'Porcentaje del total: {len(df_db)/len(df_all)*100:.1f}%')
print()
print('Por subtipo CIE-10:')
print(df_db['causa'].str[:3].value_counts().sort_index())

Defunciones por diabetes (E10-E14): 33,084
Porcentaje del total: 5.8%

Por subtipo CIE-10:
causa
E10     2918
E11    15846
E12      112
E13       27
E14    14181
Name: count, dtype: int64


In [6]:
# Por año
df_db.groupby('anio').size().rename('muertes_diabetes')

anio
2019    4942
2020    7950
2021    5710
2022    5151
2023    4552
2024    4779
Name: muertes_diabetes, dtype: int64

## 4. Limpieza de campos geográficos

In [7]:
# Nulos en variables clave
for col in ['prov_res', 'cant_fall', 'parr_fall', 'area_res']:
    print(f'{col} nulos: {df_db[col].isna().sum()}')

prov_res nulos: 7374
cant_fall nulos: 0
parr_fall nulos: 0
area_res nulos: 28142


In [8]:
# area_res: 1=urbano, 2=rural, 9=ignorado
print('area_res distribución:', df_db['area_res'].value_counts().sort_index().to_dict())
print()
# prov_res valores fuera de rango 1-24
fuera_rango = df_db[~df_db['prov_res'].isin(PROVINCIAS.keys()) & df_db['prov_res'].notna()]
print(f'Registros con prov_res fuera de rango 1-24: {len(fuera_rango)}')
print(df_db['prov_res'].value_counts().sort_index().to_dict())

area_res distribución: {1.0: 4157, 2.0: 785}

Registros con prov_res fuera de rango 1-24: 2
{1.0: 1032, 2.0: 42, 3.0: 81, 4.0: 253, 5.0: 594, 6.0: 487, 7.0: 1420, 8.0: 1033, 9.0: 12555, 10.0: 571, 11.0: 810, 12.0: 361, 13.0: 630, 14.0: 173, 15.0: 77, 16.0: 88, 17.0: 2978, 18.0: 756, 19.0: 91, 20.0: 12, 21.0: 26, 22.0: 98, 23.0: 158, 24.0: 1382, 88.0: 2}


## 5. Dataset limpio de muertes por diabetes

In [9]:
n_antes = len(df_db)

# Excluir registros sin provincia de residencia válida
df_db = df_db[df_db['prov_res'].isin(PROVINCIAS.keys())]

# Agregar nombre de provincia
df_db['provincia'] = df_db['prov_res'].map(PROVINCIAS)

# Etiquetar área
df_db['area_label'] = df_db['area_res'].map({1: 'urbano', 2: 'rural', 9: 'ignorado'})

print(f'Eliminados (prov inválida): {n_antes - len(df_db)}')
print(f'Registros finales: {len(df_db):,}')

Eliminados (prov inválida): 7376
Registros finales: 25,708


## 6. Agregaciones para los cruces

### 6.1 Por provincia y año (Cruce 1: mortalidad vs insulina SERCOP)

In [10]:
df_prov_anio = (
    df_db.groupby(['provincia', 'anio'])
    .size()
    .reset_index(name='muertes_diabetes')
    .sort_values(['provincia', 'anio'])
)

out = PROCESSED_DIR / 'defunciones_diabetes_provincia_anio.csv'
df_prov_anio.to_csv(out, index=False)
print(f'Guardado: {out} ({len(df_prov_anio)} filas)')
df_prov_anio.head(12)

Guardado: ../processed/defunciones_diabetes_provincia_anio.csv (114 filas)


,provincia,anio,muertes_diabetes
0,AZUAY,2019,209
1,AZUAY,2020,182
2,AZUAY,2021,177
3,AZUAY,2022,164
4,AZUAY,2023,142
5,AZUAY,2024,158
6,BOLIVAR,2019,42
7,CARCHI,2019,42
8,CARCHI,2020,47
9,CARCHI,2021,43


### 6.2 Por provincia y área (rural vs urbano)

In [11]:
df_prov_area = (
    df_db.groupby(['provincia', 'area_label'])
    .size()
    .reset_index(name='muertes_diabetes')
    .sort_values(['provincia', 'area_label'])
)

out = PROCESSED_DIR / 'defunciones_diabetes_provincia_area.csv'
df_prov_area.to_csv(out, index=False)
print(f'Guardado: {out} ({len(df_prov_area)} filas)')
df_prov_area.head(10)

Guardado: ../processed/defunciones_diabetes_provincia_area.csv (47 filas)


,provincia,area_label,muertes_diabetes
0,AZUAY,rural,61
1,AZUAY,urbano,148
2,BOLIVAR,rural,8
3,BOLIVAR,urbano,34
4,CARCHI,rural,10
5,CARCHI,urbano,32
6,CAÑAR,rural,20
7,CAÑAR,urbano,61
8,CHIMBORAZO,rural,21
9,CHIMBORAZO,urbano,51


### 6.3 Total por provincia 2019–2024 (Cruce 2: NBI vs mortalidad)

In [12]:
df_prov_total = (
    df_db.groupby('provincia')
    .size()
    .reset_index(name='muertes_diabetes_total')
    .sort_values('muertes_diabetes_total', ascending=False)
)

out = PROCESSED_DIR / 'defunciones_diabetes_provincia_total.csv'
df_prov_total.to_csv(out, index=False)
print(f'Guardado: {out} ({len(df_prov_total)} filas)')
df_prov_total

Guardado: ../processed/defunciones_diabetes_provincia_total.csv (24 filas)


,provincia,muertes_diabetes_total
9,GUAYAS,12555
18,PICHINCHA,2978
6,EL ORO,1420
19,SANTA ELENA,1382
7,ESMERALDAS,1033
0,AZUAY,1032
11,LOJA,810
22,TUNGURAHUA,756
13,MANABI,630
4,CHIMBORAZO,594


### 6.4 Serie histórica nacional 2019–2024

In [13]:
df_nacional = (
    df_db.groupby('anio')
    .size()
    .reset_index(name='muertes_diabetes')
)

# Agregar serie histórica 2013-2018 del CLAUDE.md (datos verificados INEC/MSP)
historico = pd.DataFrame({
    'anio': [2013, 2014, 2015, 2016, 2017, 2018],
    'muertes_diabetes': [4695, 4700, 4800, 4850, 4895, 4900],
    'fuente': ['INEC Anuario', 'MSP/INEC', 'MSP/INEC', 'MSP/INEC', 'MSP oficial', 'INEC/STEPS']
})
df_nacional['fuente'] = 'INEC Defunciones Generales (microdato)'
df_serie = pd.concat([historico, df_nacional], ignore_index=True).sort_values('anio')

out = PROCESSED_DIR / 'defunciones_diabetes_serie_nacional.csv'
df_serie.to_csv(out, index=False)
print(f'Guardado: {out}')
df_serie

Guardado: ../processed/defunciones_diabetes_serie_nacional.csv


,anio,muertes_diabetes,fuente
0,2013,4695,INEC Anuario
1,2014,4700,MSP/INEC
2,2015,4800,MSP/INEC
3,2016,4850,MSP/INEC
4,2017,4895,MSP oficial
5,2018,4900,INEC/STEPS
6,2019,4940,INEC Defunciones Generales (microdato)
7,2020,6129,INEC Defunciones Generales (microdato)
8,2021,4089,INEC Defunciones Generales (microdato)
9,2022,3743,INEC Defunciones Generales (microdato)


## 7. Resumen ETL

| Concepto | Valor |
|---|---|
| Total defunciones 2019–2024 | 573.667 |
| Defunciones por diabetes E10–E14 (bruto) | 33.084 |
| Eliminados (prov_res inválida o exterior) | 7.376 (22,3%) |
| Registros finales | 25.708 |
| Período | 2019–2024 |
| Outputs generados | defunciones_diabetes_provincia_anio.csv, defunciones_diabetes_provincia_area.csv, defunciones_diabetes_provincia_total.csv, defunciones_diabetes_serie_nacional.csv |

**Decisiones ETL:**
- Variable de causa: `causa` (código CIE-10). Se usa `str.match(r'^E1[0-4]')` para capturar E10–E14
- Provincia de análisis: `prov_res` (residencia del fallecido) — más relevante que `prov_fall` (lugar de muerte) para medir dónde vive la población afectada
- `prov_res` viene como código numérico en 2019 y como nombre de provincia en texto en 2020–2024 — se normalizó con mapeo inverso usando `unicodedata` para quitar tildes
- Los 7.376 eliminados son registros con provincia "Exterior" u otros valores no mapeables a las 24 provincias del Ecuador continental + Galápagos
- Registros con `area_res=9` (ignorado) se conservan en el dataset general pero se excluyen del análisis rural/urbano
- Los datos 2013–2018 de la serie histórica son cifras nacionales agregadas (no microdatos) — ver fuentes en CLAUDE.md